# CS 3110/5110: Data Privacy
## Homework 7

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
import random
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def laplace_mech_vec(vec, sensitivity, epsilon):
    return [v + np.random.laplace(loc=0, scale=sensitivity / epsilon) for v in vec]

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Range Queries

A *range query* counts the number of rows in the dataset which have a value lying in a given range. For example, "how many participants are between the ages of 21 and 33?" is a range query. A *workload* of range queries is just a list of range queries. The code below generates 100 random range queries over ages in the adult dataset.

In [2]:
def range_query(df, col, a, b):
    return len(df[(df[col] >= a) & (df[col] < b)])

random_lower_bounds = [random.randint(1, 70) for _ in range(100)]
random_workload = [(lb, random.randint(lb, 100)) for lb in random_lower_bounds]
real_answers = [range_query(adult, 'Age', lb, ub) for (lb, ub) in random_workload]
print('First 5 queries (the queries - not the answers!): ', random_workload[:5])
print('First 5 query *answers*:', real_answers[:5])

First 5 queries (the queries - not the answers!):  [(11, 83), (31, 39), (7, 72), (59, 64), (38, 85)]
First 5 query *answers*: [32494, 6936, 32093, 1455, 15829]


## Question 1 (10 points)

Write code to answer a workload of range queries using `laplace_mech` and sequential composition. Your solution should have a **total privacy cost of epsilon**.

In [ ]:
def workload_laplace(workload, epsilon):

    epsilon = epsilon/len(workload)
    return [laplace_mech( range_query(adult,'Age',lower, upper), 1, epsilon) for (lower, upper) in workload]

print('First 4 answers:', workload_laplace(random_workload, 1.0)[:4])

First 4 answers: [32557.15692112605, 7095.65089591926, 31967.635146986184, 1603.9428364459154]


In [26]:
errors = [abs(r_a - l_a) for (r_a, l_a) in zip(real_answers, workload_laplace(random_workload, 1.0))]
print('Average absolute error:', np.mean(errors))
assert np.mean(errors) > 50
assert np.mean(errors) < 200

Average absolute error: 98.49009599979738


## Question 2 (10 points)

Write code to answer a workload using `laplace_mech_vec` - the version of the Laplace mechanism for **vector-valued** queries. Your solution should *not* use sequential composition, and should have a total privacy cost of `epsilon`.

*Hint*: remember to use L1 global sensitivity.

In [36]:
def workload_laplace_vec(workload, epsilon):
    
    sensitivity = len(workload)
    vec = [range_query(adult,'Age',lower, upper) for (lower,upper) in workload]
    return laplace_mech_vec(vec, sensitivity, epsilon)

print('First 4 answers:', workload_laplace_vec(random_workload, 1.0)[:4])

First 4 answers: [32160.905623439106, 6803.652721690133, 32284.288586641313, 1463.5315222195643]


In [39]:
errors = [abs(r_a - l_a) for (r_a, l_a) in zip(real_answers, workload_laplace_vec(random_workload, 1.0))]
print('Average absolute error:', np.mean(errors))
assert np.mean(errors) > 50
assert np.mean(errors) < 200

Average absolute error: 112.27103445093117


## Question 3 (10 points)

In 2-5 sentences, answer the following:
- Did the two solutions differ in terms of their accuracy?
- How do they differ in terms of their use of composition properties of differential privacy?

1) Both solutions have similar accuracy
2) Fist uses sequential composition aplying e pice by pice nad second applies the e all at once

## Question 4 (10 points)

Write code to answer a workload using `gaussian_mech_vec` - the version of the Gaussian mechanism for vector-valued queries. Your solution should not use sequential composition, should satisfy $(\epsilon, \delta)$-differential privacy, and should have a total privacy cost of (`epsilon`, `delta`).

*Hint*: remember to use L2 sensitivity.

In [40]:
def workload_gaussian_vec(workload, epsilon, delta):

    sensitivity = np.sqrt( len(workload))
    vec = [range_query(adult,'Age',lower, upper) for(lower, upper) in workload]
    return gaussian_mech_vec(vec, sensitivity, epsilon, delta)

print('First 4 answers:', workload_gaussian_vec(random_workload, 1.0, 1e-5)[:4])

First 4 answers: [32507.568074930885, 6921.57379903959, 32046.492767152366, 1408.8995854497248]


In [43]:
errors = [abs(r_a - l_a) for (r_a, l_a) in zip(real_answers, workload_gaussian_vec(random_workload, 1.0, 1e-5))]
print('Average absolute error:', np.mean(errors))
assert np.mean(errors) > 10
assert np.mean(errors) < 100

Average absolute error: 40.00238507468757


## Question 5 (10 points)

In 2-5 sentences, answer the following:
- Of your solutions in questions 1-3, which ones rely on *sequential composition*?
- Which solution offers the best accuracy?
- Why does this particular solution yield the best accuracy?

1) fisrt relies on sequential composition
2) third solution has the best accuracy
3) because it uses L2 sensitivity

## Question 6 (10 points)

Re-implement your solution to question 3 using *Rényi differential privacy*. Your solution should satisfy $(\alpha, \bar\epsilon)$-RDP.

*Hint*: see the "variants" chapter in the textbook.

In [52]:
def workload_gaussian_vec_RDP(workload, alpha, epsilon_bar):

    sensitivity = np.sqrt(len(workload))
    vec = [range_query(adult,'Age',lower, upper) for(lower, upper) in workload]

    sigma = np.sqrt((sensitivity**2 * alpha) / (2 * epsilon_bar))
    return [v + np.random.normal(loc=0, scale=sigma) for v in vec]

print('First 4 answers:', workload_gaussian_vec_RDP(random_workload, 1.0, 1e-5)[:4])

First 4 answers: [29906.43311442779, 9766.493698172528, 28299.546296639437, 214.17172571578203]


In [53]:
# TEST CASE
errors = [abs(r_a - l_a) for (r_a, l_a) in zip(real_answers, workload_gaussian_vec_RDP(random_workload, 5, 0.1))]
print('Average absolute error:', np.mean(errors))
assert np.mean(errors) > 10
assert np.mean(errors) < 100

Average absolute error: 34.691475162829654


## Question 7 (10 points)

Implement a function `convert_RDP_ED` to convert from the $(\alpha, \bar\epsilon)$ of Rényi differential privacy to the $(\epsilon, \delta)$ of approximate differential privacy. Your function should also take the desired value of $\delta$.

In [ ]:
def convert_RDP_ED(alpha, epsilon_bar, delta):
    return epsilon_bar + np.log(1/delta)/(alpha - 1)

convert_RDP_ED(5, 0.1, 1e-5)

np.float64(2.9782313662425572)

In [55]:
# TEST CASE
assert convert_RDP_ED(5, 0.1, 1e-5) == 2.9782313662425572
assert convert_RDP_ED(40, 0.1, 1e-5) == 0.39520321705051864
assert convert_RDP_ED(500, 1.0, 1e-5) == 1.02307199491978
assert convert_RDP_ED(40, 1.0, 1e-5) == 1.2952032170505188

## Question 8 (10 points)

In 2-5 sentences, answer the following:
- Try various values for `alpha` and `epsilon_bar` in `convert_RDP_ED`. At what values do you observe an $(\epsilon, \delta)$ value around $(1.0, 10^{-5})$?
- Try these values for `alpha` and `epsilon_bar` in `workload_gaussian_vec_RDP`. How does the error compare to using `workload_gaussian_vec`?
- Is it useful to use Rényi differential privacy to answer workloads of range queries? Or is regular $(\epsilon, \delta)$-differential privacy just as good?

- workload_gaussian_vec error is lower then workload_gaussian_vec_RDP
- Regular $(\epsilon, \delta)$-differential privacy shows better results